# Voice Quality Dashboard

Runs a batch of voice quality analyses across a cohort of audio files and
produces seaborn box-plots with clinical threshold lines.

**Metrics analysed:** AVQI, CPP, HNR (multi-band), Jitter, Shimmer, DSI

**Requirements:** `pip install requests python-dotenv pandas seaborn matplotlib`

**Setup:** Copy `.env.example` to `.env` at the repo root and add your `VOCAMETRIX_API_KEY`.

In [ ]:
import os
import json
import glob
import concurrent.futures
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dotenv import load_dotenv

load_dotenv('../.env')
API_KEY = os.environ['VOCAMETRIX_API_KEY']
BASE_URL = 'https://platform.vocametrix.com'
HEADERS = {'X-API-Key': API_KEY}
print('API key loaded:', API_KEY[:6] + '...')

## Configuration

Point `AUDIO_FOLDER` at a directory of WAV files. Each file should be a
sustained vowel /a/ recording. The filename (without extension) is used as
the patient/subject identifier.

In [ ]:
AUDIO_FOLDER = '../test_audio'  # Change this to your cohort folder
MAX_WORKERS = 3  # Parallel uploads (stay within rate limit)

wav_files = sorted(glob.glob(os.path.join(AUDIO_FOLDER, '*.wav')))
print(f'Found {len(wav_files)} WAV files:')
for f in wav_files:
    print(f'  {os.path.basename(f)}')

## Upload and analyse

In [ ]:
def assign_file_id(audio_path):
    with open(audio_path, 'rb') as f:
        r = requests.post(f'{BASE_URL}/api/assignFileId', headers=HEADERS,
                          files={'audio': f}, data={'email': 'user@example.com'})
    r.raise_for_status()
    return r.json()['fileId']

def analyse_file(audio_path):
    name = os.path.splitext(os.path.basename(audio_path))[0]
    file_id = assign_file_id(audio_path)

    def get(endpoint):
        r = requests.get(f'{BASE_URL}{endpoint}', headers=HEADERS, params={'svFileId': file_id})
        r.raise_for_status()
        return r.json()

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
        futures = {
            'avqi': pool.submit(get, '/api/calculate-avqi'),
            'dsi':  pool.submit(get, '/api/calculate-dsi'),
            'cpp':  pool.submit(get, '/api/calculate-cpp'),
            'hnr':  pool.submit(get, '/api/calculate-hnr'),
            'js':   pool.submit(get, '/api/calculate-jitter-shimmer'),
        }
        done = {k: v.result() for k, v in futures.items()}

    js = done['js']
    return {
        'subject':  name,
        'AVQI':     done['avqi'].get('AVQI'),
        'DSI':      done['dsi'].get('DSI'),
        'CPP':      done['cpp'].get('CPP_MEAN', done['cpp'].get('CPP')),
        'HNR25':    done['hnr'].get('HNR25', done['hnr'].get('HNR')),
        'Jitter_%': js.get('JITTER_LOCAL_PERCENT', js.get('JITTER')),
        'Shimmer_%':js.get('SHIMMER_LOCAL_PERCENT', js.get('SHIMMER')),
    }

print('Analysing cohort...')
rows = []
for fp in wav_files:
    try:
        row = analyse_file(fp)
        rows.append(row)
        print(f"  ✓ {row['subject']}  AVQI={row['AVQI']}")
    except Exception as e:
        print(f'  ✗ {os.path.basename(fp)}: {e}')

df = pd.DataFrame(rows)
df

## Clinical threshold reference

| Metric | Normal range | Source |
|--------|-------------|--------|
| AVQI | < 2.97 | Maryn & Weenink (2015) |
| DSI | > 1.6 | Wuyts et al. (2000) |
| CPP | > 4.5 dB | Heman-Ackah et al. |
| Jitter (local) | < 1.04% | Teixeira & Gonçalves (2014) |
| Shimmer (local) | < 3.81% | Teixeira & Gonçalves (2014) |

In [ ]:
# Clinical thresholds (dysphonia boundary)
THRESHOLDS = {
    'AVQI':      {'value': 2.97,  'direction': 'above', 'label': 'AVQI ≥ 2.97 = dysphonic'},
    'DSI':       {'value': 1.6,   'direction': 'below', 'label': 'DSI ≤ 1.6 = dysphonic'},
    'CPP':       {'value': 4.5,   'direction': 'below', 'label': 'CPP < 4.5 dB = abnormal'},
    'Jitter_%':  {'value': 1.04,  'direction': 'above', 'label': 'Jitter ≥ 1.04% = elevated'},
    'Shimmer_%': {'value': 3.81,  'direction': 'above', 'label': 'Shimmer ≥ 3.81% = elevated'},
}

metrics = [m for m in ['AVQI', 'DSI', 'CPP', 'HNR25', 'Jitter_%', 'Shimmer_%'] if m in df.columns]
n = len(metrics)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
if n == 1: axes = [axes]

for ax, metric in zip(axes, metrics):
    col_data = df[metric].dropna()
    ax.boxplot(col_data, patch_artist=True,
               boxprops=dict(facecolor='#cce5ff', color='#004085'),
               medianprops=dict(color='#004085', linewidth=2))

    # Individual points
    import numpy as np
    x_jitter = np.random.uniform(0.85, 1.15, size=len(col_data))
    ax.scatter(x_jitter, col_data, color='#004085', alpha=0.6, zorder=5, s=30)

    # Threshold line
    if metric in THRESHOLDS:
        t = THRESHOLDS[metric]
        ax.axhline(t['value'], color='red', linestyle='--', linewidth=1.5, alpha=0.8)
        ax.text(1.35, t['value'], t['label'], va='center', fontsize=7, color='red')

    ax.set_title(metric, fontweight='bold')
    ax.set_xticks([])
    ax.set_ylabel('Value')
    ax.grid(True, axis='y', alpha=0.3)

fig.suptitle('Voice Quality Metrics — Cohort Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('voice_quality_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved voice_quality_dashboard.png')

## Export results

In [ ]:
df.to_csv('voice_quality_cohort.csv', index=False)
print('Results saved to voice_quality_cohort.csv')
df.describe()